# ANKAA quickstart (Python)

This notebook opens a small Binance BTCUSDT capture (60 frames of trades + depth diffs, a few seconds of market data) that ships with the
repository, decodes it in-process and runs five SQL queries on the resulting tables. Results come
back as `pyarrow.Table` through the Arrow C Data Interface and are converted to pandas / polars.

Install first (from the repository root):

```sh
cd crates/ankaa-py && pip install maturin pyarrow pandas polars matplotlib && maturin develop --release
```

In [ ]:
from pathlib import Path

import ankaa

print("ankaa", ankaa.__version__)

# the fixture lives in the repository; adjust the path if you copied the notebook elsewhere
root = Path.cwd()
while not (root / "crates").exists() and root.parent != root:
    root = root.parent
capture = root / "crates/ankaa-feeds/tests/fixtures/binance-btcusdt-trade-depth-60.ankc"
capture

## Open the capture

`open_capture` runs the same path as `ankaad replay`: every frame of the `.ankc` file goes through
the Binance decoder, trades land in `trades`, depth diffs in `book_deltas`, and the top of book
after every exchange message is sampled into `quotes`. `stats()` reports what the decoder saw.

In [ ]:
ds = ankaa.open_capture(str(capture))
print(ds.tables())
ds.stats()

Prices and sizes are fixed-point `Int64` columns with eight decimals (field metadata `ankaa.scale = 8`),
so divide by `1e8` in SQL. `ts` is a nanosecond UTC timestamp.

In [ ]:
ds.schema("SELECT ts, price, size, side FROM trades")

## 1. Trades per second

A plain `GROUP BY` on the timestamp truncated to the second. `side` is `0` for a buy (taker bought)
and `1` for a sell.

In [ ]:
tps = ds.sql('''
    SELECT date_trunc('second', ts) AS second,
           count(*)                                   AS trades,
           sum(CASE WHEN side = 0 THEN 1 ELSE 0 END)  AS buys,
           sum(size) / 1e8                            AS qty_btc
    FROM trades
    GROUP BY second
    ORDER BY second
''').to_pandas()
tps.head(10)

## 2. OHLCV in 1 s buckets

`ohlcv(table, bucket_ns)` is a table function implemented by the engine (`ankaa-ts`): one pass over
the trades, one row per symbol and bucket. The bucket width is in nanoseconds; `1000000000` is one second.

In [ ]:
bars = ds.sql_polars('''
    SELECT bucket, open / 1e8 AS open, high / 1e8 AS high, low / 1e8 AS low, close / 1e8 AS close,
           volume / 1e8 AS volume, count
    FROM ohlcv('trades', 1000000000)
    ORDER BY bucket
''')
bars.head(10)

## 3. ASOF join: each trade with the quote in force when it happened

`asof_join(left, right, key)` matches every left row with the latest right row at or before it,
per key. Columns from the right table are prefixed with its name (`quotes_bid`, `quotes_ask`).
From that we classify trades as hitting the bid or lifting the ask. Trades before the first quote
have no match (`NULL`).

In [ ]:
asof = ds.sql('''
    SELECT ts,
           price / 1e8      AS price,
           size / 1e8       AS size,
           quotes_bid / 1e8 AS bid,
           quotes_ask / 1e8 AS ask,
           CASE WHEN quotes_bid IS NULL THEN 'no quote'
                WHEN price <= quotes_bid THEN 'at bid'
                WHEN price >= quotes_ask THEN 'at ask'
                ELSE 'inside' END AS where_
    FROM asof_join('trades', 'quotes', 'symbol')
    ORDER BY ts
''').to_pandas()
asof["where_"].value_counts()

## 4. Percentiles of trade size

`percentiles(table, column, 'q1,q2,...')` runs a UDDSketch (1 % relative error) over the column and
returns one row with `count`, `min`, `max` and one column per quantile. Values are in ticks, so
they are scaled here like the other fixed-point columns.

In [ ]:
pct = ds.sql('''
    SELECT count, min / 1e8 AS min_btc, p50 / 1e8 AS p50_btc, p90 / 1e8 AS p90_btc,
           p99 / 1e8 AS p99_btc, max / 1e8 AS max_btc
    FROM percentiles('trades', 'size', '0.5,0.9,0.99')
''').to_pandas()
pct

## 5. Top-of-book spread

`quotes` holds the best bid and ask after every exchange message that changed the top of book.
The spread in quote currency and in basis points of the mid, per second.

In [ ]:
spread = ds.sql_polars('''
    SELECT date_trunc('second', ts)                        AS second,
           count(*)                                        AS updates,
           avg((ask - bid) / 1e8)                          AS spread_usdt,
           avg((ask - bid) * 10000.0 / ((ask + bid) / 2))  AS spread_bps,
           min(bid) / 1e8                                  AS min_bid,
           max(ask) / 1e8                                  AS max_ask
    FROM quotes
    GROUP BY second
    ORDER BY second
''')
spread.head(10)

## Plot: 1 s close price and traded volume

Two panels sharing the time axis (two measures with different units get two panels, not two y-axes).

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

t = bars["bucket"].to_list()
fig, (ax_px, ax_vol) = plt.subplots(2, 1, figsize=(10, 5), sharex=True,
                                    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.1})
ax_px.plot(t, bars["close"].to_list(), color="#2a78d6", linewidth=2, marker="o", markersize=6)
ax_px.set_ylabel("close (USDT)")
ax_px.set_title("BTCUSDT, 1 s buckets", loc="left")
ax_px.ticklabel_format(axis="y", useOffset=False)
ax_vol.bar(t, bars["volume"].to_list(), width=0.8 / 86400, color="#2a78d6")
ax_vol.set_ylabel("volume (BTC)")
ax_vol.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
for ax in (ax_px, ax_vol):
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#e6e4df", linewidth=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
plt.show()

## Where next

* `ankaa.connect("host:port", token=...)` gives the same `sql()` / `tables()` / `schema()` API against a
  running `ankaad serve` over Arrow Flight (TLS and mutual TLS through `tls_ca`, `tls_cert`, `tls_key`).
* Any Arrow consumer works on the results: `ds.sql_arrow(q)` returns the native object with
  `__arrow_c_stream__`, so `duckdb`, `nanoarrow` or `pl.from_arrow` can take it directly.
* `docs/data-model.md` in the repository describes the tables; `crates/ankaa-sql` the table functions.